<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session07_document_processing_for_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۷: آماده‌سازی اسناد فارسی برای RAG

### اهداف این جلسه

- شناخت مراحل خط لوله RAG و جایگاه آماده‌سازی اسناد در آن
- خواندن فایل‌های Word و PDF و شناخت مشکلات استخراج متن فارسی از PDF
- سنجیدن کیفیت استخراج با یک معیار عددی و اصلاح خطاهای رایج
- نرمال‌سازی متن فارسی برای جستجو
- پیاده‌سازی و مقایسه روش‌های تقسیم متن (Chunking) با یک ارزیابی بازیابی

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. خط لوله RAG

در **RAG (Retrieval-Augmented Generation)** مدل زبانی به جای تکیه بر حافظه خودش، پاسخ را بر اساس اسنادی می‌سازد که برایش بازیابی کرده‌ایم. خط لوله دو بخش دارد:

**آماده‌سازی (یک بار، آفلاین):** خواندن اسناد ← پاک‌سازی و نرمال‌سازی ← تقسیم به تکه‌ها (Chunk) ← ساخت امبدینگ ← ذخیره در پایگاه داده برداری

**پاسخ‌گویی (برای هر پرسش):** امبدینگ پرسش ← بازیابی تکه‌های مرتبط ← ساخت پرامپت با تکه‌ها ← تولید پاسخ

در جلسه قبل بازیابی را دیدیم و در جلسه بعد تولید پاسخ را می‌سازیم. این جلسه به بخشی می‌پردازد که معمولاً کمتر دیده می‌شود ولی بیشترین اثر را بر کیفیت دارد. اگر متن اسناد خراب استخراج شود یا بد تقسیم شود، هیچ مدل زبانی‌ای نمی‌تواند پاسخ درست بدهد: **Garbage in, garbage out.**

</div>

In [ ]:
!pip -q install pymupdf pypdf python-docx hazm sentence-transformers
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

import re

import numpy as np
import pandas as pd
from llm_utils import download_data

docx_path, pdf_path = download_data("novinkala_handbook.docx", "novinkala_handbook.pdf")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. خواندن فایل Word

فایل‌های Word ساختار دارند: هر پاراگراف یک **سبک (Style)** دارد که عنوان‌ها را از متن عادی جدا می‌کند. این ساختار را نگه می‌داریم و به Markdown تبدیل می‌کنیم، چون بعداً برای تقسیم متن بر اساس بخش‌ها به آن نیاز داریم.

</div>

In [ ]:
from docx import Document


def read_docx(path):
    lines = []
    for paragraph in Document(path).paragraphs:
        text = paragraph.text.strip()
        if not text:
            continue
        style = paragraph.style.name
        if style == "Title" or style == "Heading 1":
            lines.append(f"# {text}")
        elif style.startswith("Heading"):
            lines.append(f"## {text}")
        else:
            lines.append(text)
    return "\n\n".join(lines)


docx_text = read_docx(docx_path)
print(len(docx_text), "characters")
print(docx_text[:600])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. خواندن PDF: جایی که مشکلات شروع می‌شود

فایل PDF برای **نمایش** طراحی شده، نه برای نگهداری متن. در PDF هر حرف یک «نقش» (Glyph) با مختصات روی صفحه است. ترتیب منطقی کلمات، جهت راست‌به‌چپ و حروف ترکیبی باید هنگام استخراج حدس زده شوند. برای متن فارسی این حدس زدن زیاد خطا دارد. دو کتابخانه رایج را مقایسه می‌کنیم:

</div>

In [ ]:
from pypdf import PdfReader
import pymupdf

pypdf_text = "\n".join(page.extract_text() for page in PdfReader(pdf_path).pages)
mupdf_text = "\n".join(page.get_text() for page in pymupdf.open(pdf_path))

print("=== pypdf ===\n" + pypdf_text[:350])
print("\n=== PyMuPDF ===\n" + mupdf_text[:350])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

خروجی pypdf ترتیب کلمات را به‌هم ریخته است. PyMuPDF خیلی بهتر است، ولی هنوز سه مشکل دارد:

1. **حرف ترکیبی «لا»** به «ال» تبدیل شده است: «کالا» ← «کاال»، «اعلام» ← «اعالم».
2. **نقطه پایان جمله** به ابتدای خط رفته است.
3. **شکستن خط** در وسط جمله‌ها رخ داده، چون هر خط صفحه یک خط جدا استخراج شده است.

### سنجیدن کیفیت استخراج

به جای نگاه کردن به چند خط، کیفیت را **اندازه می‌گیریم**: درصد کلماتی که در یک واژه‌نامه فارسی وجود دارند. کتابخانه hazm یک واژه‌نامه حدود ۲۰۰ هزار کلمه‌ای دارد. چون این واژه‌نامه شکل‌های صرفی مثل «کالاهای» را ندارد، پسوندهای رایج را هم جدا می‌کنیم.

</div>

In [ ]:
from hazm.utils import words_list

LEXICON = {w[0] if isinstance(w, tuple) else w for w in words_list()}
SUFFIXES = ("هایی", "های", "ها", "ای", "یی", "ی", "ش", "م")
WORD = re.compile(r"[\u0600-\u06FF\u200c]+")


def in_lexicon(word):
    word = word.replace("\u200c", "")
    if word in LEXICON:
        return True
    return any(word.endswith(s) and word[:-len(s)] in LEXICON for s in SUFFIXES)


def lexicon_coverage(text):
    words = WORD.findall(text)
    return sum(in_lexicon(w) for w in words) / len(words)


for name, text in [("DOCX", docx_text), ("pypdf", pypdf_text), ("PyMuPDF", mupdf_text)]:
    print(f"{name:8} lexicon coverage: {lexicon_coverage(text):.1%}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

متن Word، که مرجع ماست، هم پوشش ۱۰۰ درصد ندارد، چون نام‌های خاص و واژه‌های جدید در واژه‌نامه نیستند. ولی اختلاف PDF با آن، اندازه خرابی استخراج را نشان می‌دهد.

### اصلاح خطاهای PyMuPDF

برای حرف «لا» نمی‌توانیم همه «ال»ها را جابه‌جا کنیم، چون «ارسال» و «سال» درست‌اند. راه امن این است: کلمه‌ای را اصلاح می‌کنیم که **خودش در واژه‌نامه نیست ولی نسخه اصلاح‌شده‌اش هست**.

</div>

In [ ]:
def fix_lam_alef(word):
    if "‌" in word:                         # fix each part of a compound word separately
        return "‌".join(fix_lam_alef(part) for part in word.split("‌"))
    if "ال" not in word or in_lexicon(word):
        return word
    for i in range(len(word) - 1):
        if word[i:i + 2] == "ال":
            candidate = word[:i] + "لا" + word[i + 2:]
            if in_lexicon(candidate):
                return candidate
    return word


def clean_pdf_text(text):
    lines = []
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("."):                 # sentence-final period moved to the start
            line = line[1:].strip() + "."
        lines.append(line)
    text = "\n".join(lines)
    text = re.sub(r"(?<![.:!؟?])\n(?!\n)", " ", text)   # join lines broken inside a sentence
    return WORD.sub(lambda m: fix_lam_alef(m.group()), text)


fixed_text = clean_pdf_text(mupdf_text)
print(f"PyMuPDF + fixes lexicon coverage: {lexicon_coverage(fixed_text):.1%}\n")
print(fixed_text[:400])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**درس‌های عملی:**

- اگر سند اصلی، یعنی Word، HTML یا Markdown، در دسترس است، **هرگز از PDF استفاده نکنید.**
- اگر فقط PDF دارید، PyMuPDF را ترجیح دهید و کیفیت استخراج را با یک معیار عددی بسنجید.
- برای PDFهای اسکن‌شده که اصلاً متن ندارند، به **OCR** نیاز دارید، مثلاً Tesseract با مدل زبان فارسی (`fas`). کیفیت OCR فارسی متوسط است و حتماً باید سنجیده شود.

## ۴. نرمال‌سازی متن فارسی

یک متن فارسی ممکن است با چند شکل مختلف نوشته شود که برای انسان یکسان و برای کامپیوتر متفاوت‌اند. اگر اسناد و پرسش‌ها را **با یک روش یکسان** نرمال نکنیم، جستجوی کلیدواژه‌ای شکست می‌خورد و حتی امبدینگ‌ها کمی جابه‌جا می‌شوند.

| مشکل | نمونه | نرمال‌شده |
|---|---|---|
| ی و ک عربی | علي، كتاب | علی، کتاب |
| ارقام | ۱۲ و ١٢ و 12 | یک شکل واحد |
| نیم‌فاصله | می روم، میروم | می‌روم |
| اعراب و کشیده | کِتـــاب | کتاب |
| فاصله‌گذاری علائم | سلام ،خوبی ؟ | سلام، خوبی؟ |

کتابخانه **hazm** نرمال‌ساز آماده دارد. ارقام را به انگلیسی تبدیل می‌کنیم، چون مدل‌های زبانی و کد پایتون با آن‌ها راحت‌ترند. مهم این است که برای اسناد و پرسش‌ها یک انتخاب ثابت داشته باشیم.

</div>

In [ ]:
from hazm import Normalizer

_normalizer = Normalizer(persian_numbers=False)
_TO_ENGLISH_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def normalize(text):
    text = _normalizer.normalize(text)
    return text.translate(_TO_ENGLISH_DIGITS)


samples = ["علي ٣ تا كتاب ميخواد", "مي روم به فروشگاه ،قيمت ۲۵۰ هزار تومانه", "کِتـــاب‌ها"]
for s in samples:
    print(f"{s!r:45} -> {normalize(s)!r}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. تقسیم متن (Chunking)

چرا اسناد را تکه‌تکه می‌کنیم؟

- **محدودیت مدل امبدینگ:** E5 فقط ۵۱۲ توکن اول را می‌بیند.
- **دقت بازیابی:** بردار یک سند بلند میانگین موضوعات زیادی است و با هیچ پرسش مشخصی خیلی شبیه نمی‌شود.
- **هزینه و تمرکز:** فقط تکه‌های مرتبط وارد پرامپت می‌شوند، نه کل سند.

ولی تکه‌های خیلی کوچک هم زمینه را از دست می‌دهند: جمله «هزینه آن ۸۵ هزار تومان است» بدون جمله قبلی معلوم نیست هزینه چه چیزی است. انتخاب اندازه و روش تقسیم یک موازنه است.

متن مرجع را از فایل‌های Markdown می‌خوانیم و نرمال می‌کنیم:

</div>

In [ ]:
from llm_utils import load_store_docs

documents = {name: normalize(text) for name, text in load_store_docs().items()}
print(len(documents), "documents,", sum(len(t) for t in documents.values()), "characters")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### روش ۱: اندازه ثابت با هم‌پوشانی

ساده‌ترین روش: هر `size` کاراکتر یک تکه، با `overlap` کاراکتر مشترک بین تکه‌های پشت‌سرهم تا جمله‌ای که روی مرز افتاده کاملاً از دست نرود. عیب این روش آن است که جمله‌ها و حتی کلمات را از وسط می‌برد.

</div>

In [ ]:
def chunk_fixed(text, size=400, overlap=80):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks


example = chunk_fixed(documents["01_shipping.md"])
print(len(example), "chunks")
for c in example[:2]:
    print("---\n" + c)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### روش ۲: تقسیم بازگشتی

ایده: متن را ابتدا با «بزرگ‌ترین» جداکننده، یعنی مرز بخش‌ها، تقسیم می‌کنیم. اگر تکه‌ای هنوز بزرگ‌تر از حد مجاز بود، آن را با جداکننده کوچک‌تر بعدی، مثل پاراگراف، جمله و کلمه، تقسیم می‌کنیم. تکه‌های کوچک کنار هم را دوباره ادغام می‌کنیم تا به اندازه مطلوب نزدیک شوند. کتابخانه LangChain این روش را با نام `RecursiveCharacterTextSplitter` دارد. اینجا نسخه ساده‌اش را خودمان می‌نویسیم تا منطقش روشن شود.

</div>

In [ ]:
SEPARATORS = ["\n## ", "\n\n", "\n", ". ", "، ", " "]


def chunk_recursive(text, size=400, separators=SEPARATORS):
    if len(text) <= size:
        return [text.strip()] if text.strip() else []
    sep = next((s for s in separators if s in text), None)
    if sep is None:
        return chunk_fixed(text, size, 0)
    parts = text.split(sep)
    pieces = [parts[0]] + [sep + p for p in parts[1:]]      # keep the separator with the next piece
    chunks, current = [], ""
    for piece in pieces:
        if len(current) + len(piece) <= size:
            current += piece
            continue
        if current.strip():
            chunks.append(current.strip())
        if len(piece) > size:
            rest = separators[separators.index(sep) + 1:]
            chunks.extend(chunk_recursive(piece, size, rest))
            current = ""
        else:
            current = piece
    if current.strip():
        chunks.append(current.strip())
    return chunks


example = chunk_recursive(documents["01_shipping.md"])
print(len(example), "chunks")
for c in example[:3]:
    print("---\n" + c)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### روش ۳: آگاه از ساختار، با عنوان در هر تکه

اسناد ما عنوان سند و عنوان بخش دارند. تکه «هزینه ارسال سریع ۸۵ هزار تومان است» وقتی معنای کامل دارد که بدانیم مربوط به سند «ارسال و تحویل سفارش» و بخش «ارسال سریع» است. پس متن را بر اساس بخش‌ها تقسیم می‌کنیم و **مسیر عنوان‌ها را به ابتدای هر تکه اضافه می‌کنیم**. به این تکنیک **Contextual Chunk Headers** می‌گوییم. عنوان‌ها در متادیتا هم ذخیره می‌شوند.

</div>

In [ ]:
def chunk_by_sections(text, source, size=600):
    title = text.split("\n", 1)[0].lstrip("# ").strip()
    chunks = []
    for section in re.split(r"\n(?=## )", text):
        lines = section.strip().split("\n", 1)
        if lines[0].startswith("## "):
            heading, body = lines[0][3:].strip(), (lines[1] if len(lines) > 1 else "")
        else:
            heading, body = "", section.replace(lines[0], "", 1) if lines[0].startswith("# ") else section
        for piece in chunk_recursive(body.strip(), size):
            header = f"{title} > {heading}" if heading else title
            chunks.append({"source": source, "title": title, "section": heading,
                           "text": f"{header}\n{piece}"})
    return chunks


example = chunk_by_sections(documents["01_shipping.md"], "01_shipping.md")
print(len(example), "chunks\n")
print(example[2]["text"])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. کدام روش بهتر است؟ اندازه بگیریم

برای مقایسه روش‌ها از ۳۰ پرسش فایل `rag_eval.jsonl` استفاده می‌کنیم که سند منبع هر کدام مشخص است. برای هر روش تقسیم، همه تکه‌ها را امبد می‌کنیم و بررسی می‌کنیم آیا در ۳ نتیجه اول هر پرسش، تکه‌ای از سند درست وجود دارد یا نه. به این معیار **Hit Rate@3** می‌گوییم. در جلسه ۹ معیارهای بازیابی را دقیق‌تر بررسی می‌کنیم.

</div>

In [ ]:
import json

from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("intfloat/multilingual-e5-base")
with open(download_data("rag_eval.jsonl"), encoding="utf-8") as f:
    questions = [json.loads(line) for line in f]
query_vectors = embedder.encode([f"query: {normalize(q['question'])}" for q in questions],
                                normalize_embeddings=True)

strategies = {
    "fixed 400/80": lambda name, text: [{"source": name, "text": c} for c in chunk_fixed(text)],
    "recursive 400": lambda name, text: [{"source": name, "text": c} for c in chunk_recursive(text)],
    "sections + headers": lambda name, text: chunk_by_sections(text, name),
    "whole document": lambda name, text: [{"source": name, "text": text}],
}

rows = []
for name, splitter in strategies.items():
    chunks = [c for doc, text in documents.items() for c in splitter(doc, text)]
    vectors = embedder.encode([f"passage: {c['text']}" for c in chunks], normalize_embeddings=True)
    top3 = np.argsort(-(query_vectors @ vectors.T), axis=1)[:, :3]
    hits = [any(chunks[j]["source"] == q["source"] for j in top) for q, top in zip(questions, top3)]
    lengths = [len(embedder.tokenizer(c["text"]).input_ids) for c in chunks]
    rows.append({"strategy": name, "chunks": len(chunks), "mean_tokens": np.mean(lengths),
                 "max_tokens": max(lengths), "hit_rate@3": np.mean(hits)})

pd.DataFrame(rows).round(2)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

چند نکته در تفسیر این جدول:

- حالت «کل سند» فقط ۱۲ تکه دارد، پس Hit Rate@3 آن به‌طور طبیعی بالاست: با ۳ حدس از ۱۲ گزینه! ولی هر تکه صدها توکن است و در RAG همه آن‌ها وارد پرامپت می‌شوند. به ستون `max_tokens` هم دقت کنید: تکه‌های بلندتر از ۵۱۲ توکن برای E5 بریده می‌شوند.
- این معیار فقط سند درست را بررسی می‌کند، نه اینکه تکه بازیابی‌شده **جواب** را داشته باشد. در جلسه ۹ ارزیابی دقیق‌تری می‌سازیم.
- با ۳۰ پرسش، اختلاف یکی دو پرسش معنادار نیست، همان درس جلسه ۵.

**قاعده تجربی:** برای اسناد ساختاریافته مثل آیین‌نامه و راهنما، تقسیم بر اساس بخش با عنوان در هر تکه و اندازه ۲۰۰ تا ۵۰۰ توکن نقطه شروع خوبی است. بعد با ارزیابی روی داده خودتان تنظیمش کنید.

تکه‌های روش آخر را برای جلسه بعد ذخیره می‌کنیم:

</div>

In [ ]:
final_chunks = [c for doc, text in documents.items() for c in chunk_by_sections(text, doc)]
with open("novinkala_chunks.jsonl", "w", encoding="utf-8") as f:
    for i, c in enumerate(final_chunks):
        f.write(json.dumps({"id": f"c{i:03d}", **c}, ensure_ascii=False) + "\n")
print(len(final_chunks), "chunks saved")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۷. جمع‌بندی

- کیفیت RAG بیش از هر چیز به کیفیت آماده‌سازی اسناد بستگی دارد.
- PDF برای نمایش است، نه نگهداری متن. اگر سند اصلی هست از آن استفاده کنید. اگر نه، PyMuPDF را با اصلاحات و یک معیار کیفیت عددی به کار ببرید.
- اسناد و پرسش‌ها باید با یک روش یکسان نرمال شوند.
- تقسیم بازگشتی و آگاه از ساختار، مرز جمله‌ها و بخش‌ها را حفظ می‌کند. افزودن عنوان به تکه‌ها زمینه را نگه می‌دارد.
- اندازه و روش تقسیم را با ارزیابی بازیابی انتخاب کنید، نه با حدس.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. تابع `clean_pdf_text` را روی خروجی pypdf اجرا کنید. پوشش واژه‌نامه چقدر بهتر می‌شود؟ چرا اصلاح خروجی pypdf سخت‌تر است؟
۲. در مقایسه روش‌ها، اندازه تکه را برای روش «sections + headers» به ۲۰۰، ۴۰۰ و ۸۰۰ تغییر دهید. Hit Rate@3 و میانگین توکن چگونه تغییر می‌کنند؟
۳. روش چهارمی بسازید که متن را با `sent_tokenize` از hazm به جمله‌ها بشکند و جمله‌ها را تا رسیدن به ۴۰۰ کاراکتر کنار هم بگذارد. آن را به جدول مقایسه اضافه کنید.
۴. یک سند Word یا PDF واقعی، مثلاً آیین‌نامه آموزشی دانشگاه یا یک دفترچه راهنما، انتخاب کنید و خط لوله این جلسه را روی آن اجرا کنید. پوشش واژه‌نامه چقدر است؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** تابع `clean_pdf_text` را روی خروجی pypdf اجرا کنید. پوشش واژه‌نامه چقدر بهتر می‌شود؟ چرا اصلاح خروجی pypdf سخت‌تر است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** در مقایسه روش‌ها، اندازه تکه را برای روش «sections + headers» به ۲۰۰، ۴۰۰ و ۸۰۰ تغییر دهید. Hit Rate@3 و میانگین توکن چگونه تغییر می‌کنند؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** روش چهارمی بسازید که متن را با `sent_tokenize` از hazm به جمله‌ها بشکند و جمله‌ها را تا رسیدن به ۴۰۰ کاراکتر کنار هم بگذارد. آن را به جدول مقایسه اضافه کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** یک سند Word یا PDF واقعی، مثلاً آیین‌نامه آموزشی دانشگاه یا یک دفترچه راهنما، انتخاب کنید و خط لوله این جلسه را روی آن اجرا کنید. پوشش واژه‌نامه چقدر است؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید